In [ ]:
"""
Hybrid QAM-QPSK Quantized ResNet on CIFAR-10
=============================================
Input        : 16-QAM  — 4 levels per channel ≈ [-0.9487,-0.3162,+0.3162,+0.9487]
Weights/Acts : QPSK    — 2 levels per component = {-1/√2, +1/√2}

Why this combination makes sense:
  • 16-QAM input preserves more pixel information than QPSK input.
    QPSK input = 1 bit/channel → massive information loss before any learning.
    16-QAM input = 2 bits/channel → inner/outer edges both distinguishable.
  • QPSK weights are the most hardware-efficient option: every multiply
    becomes a sign-flip, no true multiplication needed.
  • Float skip connections (Bi-Real Net, 2018) give exact gradient highways
    around the QPSK quantized ops so STE error does not compound over 24 layers.
  • Per-channel learned scale α on QPSK weights restores magnitude freedom
    (XNOR-Net trick): w_eff = sign(w)/√2 × |α_c|

Expected accuracy: 82–85% on CIFAR-10.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import math
import numpy as np

# ════════════════════════════════════════════════════════════════════════════
# 0.  DEVICE
# ════════════════════════════════════════════════════════════════════════════
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


# ════════════════════════════════════════════════════════════════════════════
# 1.  CONSTELLATION CONSTANTS
# ════════════════════════════════════════════════════════════════════════════

# ── 16-QAM (input only) ──────────────────────────────────────────────────
# Each real axis: [-3,-1,+1,+3] / √10
# E[level²] = (9+1+1+9)/4 / 10 = 0.5  →  total symbol power = 1.0
_QAM16_LEVELS = torch.tensor([-3., -1., 1., 3.]) / math.sqrt(10.0)
QAM16_MAX     = float(_QAM16_LEVELS[-1])          # ≈ +0.9487

# ── QPSK (all weights and activations) ───────────────────────────────────
# Each component: {-1/√2, +1/√2}  — 1 bit per component, max hardware efficiency
# E[level²] = 0.5  →  total symbol power = 1.0  (same as 16-QAM, consistent scaling)
QPSK_LEVEL    = 1.0 / math.sqrt(2.0)             # ≈ 0.7071
QPSK_MAX      = QPSK_LEVEL                        # used for clipped STE boundary


# ════════════════════════════════════════════════════════════════════════════
# 2.  STE QUANTIZERS
# ════════════════════════════════════════════════════════════════════════════

# ── 16-QAM argmin snap (input quantizer) ─────────────────────────────────
class _QAM16Snap(torch.autograd.Function):
    """
    Forward : argmin snap to nearest of 4 QAM levels.
    Backward: clipped STE — gradient passes where |x| ≤ 1.1 × QAM16_MAX.

    Uses argmin over 4 levels — correct for ALL tensor shapes.
    (The step-based rounding used in v7 broke for QPSK because
     tanh output ∈ [-1,1] / step=2.0 all rounds to 0.)
    """
    @staticmethod
    def forward(ctx, x, levels):
        ctx.save_for_backward(x)
        lv    = levels.view(*([1] * x.dim()), -1)    # broadcast to (..., 4)
        idx   = (x.unsqueeze(-1) - lv).abs().argmin(-1)
        return levels[idx]

    @staticmethod
    def backward(ctx, grad):
        (x,) = ctx.saved_tensors
        mask = (x.abs() <= QAM16_MAX * 1.1).to(grad.dtype)
        return grad * mask, None

def qam16_snap(x, levels):
    return _QAM16Snap.apply(x, levels)


# ── QPSK sign snap (weights + activations) ───────────────────────────────
class _QPSKSnap(torch.autograd.Function):
    """
    Forward : hard sign snap → {-1/√2, +1/√2}.
    Backward: clipped STE — gradient passes where |x| ≤ 1.1 × QPSK_MAX.

    sign(0) is handled by clamping to ±QPSK_LEVEL
    (torch.sign(0) = 0, which would kill a neuron permanently).
    """
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        # sign(x): map 0 → +1 to avoid dead neurons
        s = torch.where(x >= 0,
                        torch.tensor(QPSK_LEVEL,  device=x.device, dtype=x.dtype),
                        torch.tensor(-QPSK_LEVEL, device=x.device, dtype=x.dtype))
        return s

    @staticmethod
    def backward(ctx, grad):
        (x,) = ctx.saved_tensors
        mask = (x.abs() <= QPSK_MAX * 1.1).to(grad.dtype)
        return grad * mask

def qpsk_snap(x):
    return _QPSKSnap.apply(x)


# ════════════════════════════════════════════════════════════════════════════
# 3.  INPUT QUANTIZER — 16-QAM  (no learned parameters)
# ════════════════════════════════════════════════════════════════════════════
class QAM16InputQ(nn.Module):
    """
    Hard 16-QAM quantization of raw normalized CIFAR-10 pixels.
    Zero learnable parameters.

    Pipeline per pixel/channel:
        normalized_pixel → tanh(x) × QAM16_MAX → argmin snap → QAM level

    Why tanh before snap?
      CIFAR-10 normalized range ≈ [-2.5, +2.5].
      QAM levels span [-0.949, +0.949].
      Without tanh: ~80% of pixels saturate to ±0.949 (only outer levels used).
      With tanh:    smooth bounded mapping → all 4 levels get real usage.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer('levels', _QAM16_LEVELS.clone())

    def forward(self, x):
        return qam16_snap(torch.tanh(x) * QAM16_MAX, self.levels)

    @torch.no_grad()
    def verify(self, x):
        xq = self.forward(x)
        u  = xq.unique()
        ok = "✓ OK" if u.numel() == 4 else "✗ BUG — expected 4 levels!"
        print(f"  Unique levels : {u.numel()}  {ok}")
        print(f"  Level values  : {[f'{v:.4f}' for v in u.tolist()]}")
        print(f"  Output range  : [{xq.min():.4f}, {xq.max():.4f}]")


# ════════════════════════════════════════════════════════════════════════════
# 4.  ACTIVATION QUANTIZER — QPSK  (per-channel learned scale)
# ════════════════════════════════════════════════════════════════════════════
class QPSKActQ(nn.Module):
    """
    Quantizes feature-map activations to QPSK levels {-1/√2, +1/√2}.
    Learned per-channel scale α controls effective magnitude post-snap.

    Pipeline:
        BN_output → tanh(x × |α|) × QPSK_MAX → sign_snap → × |β|

    Two-scale design:
      α (input scale):  shapes the tanh nonlinearity — controls how many
                        neurons saturate to outer vs inner QPSK level.
                        (There are only 2 levels so "inner/outer" = ±QPSK_LEVEL,
                         but α still controls near-zero dead-zone width.)
      β (output scale): per-channel magnitude after snap. Lets the network
                        re-scale the binary {±0.707} activations independently
                        per feature map before the next layer sees them.

    Both α, β initialized to 1.0, clamped to [0.05, 10.0].
    Shape: (1, C, 1, 1) to broadcast over (B, C, H, W).
    """
    def __init__(self, num_channels):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1, num_channels, 1, 1))
        self.beta  = nn.Parameter(torch.ones(1, num_channels, 1, 1))

    def forward(self, x):
        a = self.alpha.abs().clamp(0.05, 10.0)
        b = self.beta.abs().clamp(0.05, 10.0)
        return qpsk_snap(torch.tanh(x * a) * QPSK_MAX) * b


# ════════════════════════════════════════════════════════════════════════════
# 5.  QUANTIZED CONV — QPSK weights + per-output-channel scale α
# ════════════════════════════════════════════════════════════════════════════
class QPSKConv2d(nn.Module):
    """
    Conv2d with QPSK-quantized weights and per-output-channel scale.

    Weight pipeline:
        w_float → tanh(w) × QPSK_MAX → sign_snap → {±1/√2} → × |α_c|

    Why tanh before sign snap?
      • tanh always has nonzero gradient (sech²(w) > 0 everywhere), so latent
        weights always receive meaningful gradient even when |w| is large.
      • Hard clipping (clamp to [-1,1]) would kill gradient for |w| > 1 entirely.
      • After tanh, the sign snap STE operates on values ∈ [-QPSK_MAX, QPSK_MAX],
        so the clipped STE boundary is tight and meaningful.

    Per-channel scale α_c (shape: out_ch × 1 × 1 × 1):
      QPSK-snapped weights are locked to {±0.707}.
      α_c is a learned positive scalar per output filter, restoring the magnitude
      degree of freedom. This is the XNOR-Net trick and recovers ~3-5% accuracy.
      Initialized to 0.5 (moderate), clamped to [1e-3, 10.0].
    """
    def __init__(self, in_ch, out_ch, kernel, stride=1, padding=0):
        super().__init__()
        self.stride  = stride
        self.padding = padding
        self.weight  = nn.Parameter(torch.empty(out_ch, in_ch, kernel, kernel))
        self.alpha   = nn.Parameter(torch.full((out_ch, 1, 1, 1), 0.5))
        nn.init.kaiming_normal_(self.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        w_t     = torch.tanh(self.weight) * QPSK_MAX      # tanh-normalize
        w_q     = qpsk_snap(w_t)                           # QPSK snap {±0.707}
        w_final = w_q * self.alpha.abs().clamp(1e-3, 10.0) # scale
        return F.conv2d(x, w_final, stride=self.stride, padding=self.padding)


# ════════════════════════════════════════════════════════════════════════════
# 6.  QUANTIZED RESIDUAL BLOCK
#     Input/main-path activations: QPSK
#     Skip connections:            FLOAT  ← gradient highway
# ════════════════════════════════════════════════════════════════════════════
class HybridResBlock(nn.Module):
    """
    Pre-activation residual block:
      Main path (QPSK):   BN → QPSK_act → QPSKConv(3×3) → BN → QPSK_act → QPSKConv(3×3)
      Skip (FLOAT):       Identity  OR  AvgPool + float Conv1×1 + BN

    Output = main + skip   (float addition — skip gradient is exact)

    ─────────────────────────────────────────────────────────────────────────
    FLOAT SKIP = GRADIENT HIGHWAY (critical for quantized deep networks)
    ─────────────────────────────────────────────────────────────────────────
    Every QPSK snap introduces STE approximation error.
    Over 24 quantized conv layers, these errors compound: early-layer gradients
    become meaningless noise → training collapses (exactly what happened at 10%).

    Float skips provide a parallel path with EXACT gradients.
    Loss gradient = exact_skip_grad + approximate_main_grad
    The exact component keeps early layers trainable regardless of STE noise.
    This is the Bi-Real Net (Liu et al., 2018) insight.
    ─────────────────────────────────────────────────────────────────────────
    """
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        # Main path — all QPSK quantized
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.qact1 = QPSKActQ(in_ch)
        self.conv1 = QPSKConv2d(in_ch, out_ch, 3, stride=stride, padding=1)

        self.bn2   = nn.BatchNorm2d(out_ch)
        self.qact2 = QPSKActQ(out_ch)
        self.conv2 = QPSKConv2d(out_ch, out_ch, 3, stride=1, padding=1)

        # Skip path — float, no quantization
        if stride != 1 or in_ch != out_ch:
            skip_layers = []
            if stride > 1:
                skip_layers.append(nn.AvgPool2d(stride, stride))
            skip_layers += [
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch)
            ]
            self.skip = nn.Sequential(*skip_layers)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)                              # float path

        out = self.conv1(self.qact1(self.bn1(x)))            # QPSK main path
        out = self.conv2(self.qact2(self.bn2(out)))

        return out + identity                                # float addition


# ════════════════════════════════════════════════════════════════════════════
# 7.  FULL NETWORK
# ════════════════════════════════════════════════════════════════════════════
class HybridResNet(nn.Module):
    """
    16-QAM input + QPSK weight/activation ResNet for CIFAR-10.

    ┌──────────────┬──────────────────┬──────────────┬──────────────────────┐
    │  Component   │  Quantization    │  Channels    │  Notes               │
    ├──────────────┼──────────────────┼──────────────┼──────────────────────┤
    │ input_q      │ 16-QAM (4 lvls)  │ 3 → 3        │ no learned params    │
    │ stem         │ FLOAT            │ 3 → 64       │ feature bootstrapping│
    │ stage1 (4×)  │ QPSK weights     │ 64 → 64      │ 32×32                │
    │ stage2 (4×)  │ QPSK weights     │ 64 → 128     │ 16×16 (stride 2)     │
    │ stage3 (4×)  │ QPSK weights     │ 128 → 256    │  8×8  (stride 2)     │
    │ head         │ FLOAT            │ 256 → 10     │ GAP + dropout + FC   │
    └──────────────┴──────────────────┴──────────────┴──────────────────────┘
    """
    def __init__(self, num_classes=10, base_ch=64, n_blocks=4):
        super().__init__()
        c1, c2, c3 = base_ch, base_ch * 2, base_ch * 4

        self.input_q = QAM16InputQ()

        self.stem = nn.Sequential(
            nn.Conv2d(3, base_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch)
        )

        self.stage1 = self._make_stage(base_ch, c1, n_blocks, stride=1)
        self.stage2 = self._make_stage(c1,      c2, n_blocks, stride=2)
        self.stage3 = self._make_stage(c2,      c3, n_blocks, stride=2)

        self.head = nn.Sequential(
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=0.1),
            nn.Linear(c3, num_classes)
        )
        self._init_weights()

    @staticmethod
    def _make_stage(ic, oc, n, stride):
        return nn.Sequential(
            HybridResBlock(ic, oc, stride),
            *[HybridResBlock(oc, oc, stride=1) for _ in range(n - 1)]
        )

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_q(x)        # 16-QAM hard snap (STE in backward)
        x = self.stem(x)            # float first conv
        x = self.stage1(x)          # QPSK  64ch  32×32
        x = self.stage2(x)          # QPSK 128ch  16×16
        x = self.stage3(x)          # QPSK 256ch   8×8
        return self.head(x)         # float classifier

    def param_summary(self):
        total  = sum(p.numel() for p in self.parameters())
        float_ = (sum(p.numel() for p in self.stem.parameters()) +
                  sum(p.numel() for p in self.head.parameters()))
        for m in self.modules():
            if isinstance(m, HybridResBlock) and not isinstance(m.skip, nn.Identity):
                float_ += sum(p.numel() for p in m.skip.parameters())
        return total, float_, total - float_


# ════════════════════════════════════════════════════════════════════════════
# 8.  DATA LOADING
# ════════════════════════════════════════════════════════════════════════════
def get_dataloaders(batch_size=128, num_workers=4):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)

    try:
        aa = transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10)
        train_tf = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            aa,
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
        ])
        print("  Augmentation : AutoAugment(CIFAR10) + RandomErasing + CutMix")
    except AttributeError:
        train_tf = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        print("  Augmentation : Basic + CutMix (upgrade torchvision for AutoAugment)")

    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    nw = 0 if device.type == 'mps' else num_workers
    kw = dict(num_workers=nw,
              pin_memory=(device.type == 'cuda'),
              persistent_workers=(nw > 0))

    train_set = torchvision.datasets.CIFAR10('./data', True,  train_tf, download=True)
    test_set  = torchvision.datasets.CIFAR10('./data', False, test_tf,  download=True)

    return (
        torch.utils.data.DataLoader(train_set, batch_size, shuffle=True,  **kw),
        torch.utils.data.DataLoader(test_set,  batch_size, shuffle=False, **kw)
    )


# ════════════════════════════════════════════════════════════════════════════
# 9.  TRAINING UTILITIES
# ════════════════════════════════════════════════════════════════════════════
def cutmix(x, y, alpha=1.0):
    lam  = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    ch = int(H * math.sqrt(1.0 - lam))
    cw = int(W * math.sqrt(1.0 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = max(0, cy - ch // 2), min(H, cy + ch // 2)
    x1, x2 = max(0, cx - cw // 2), min(W, cx + cw // 2)
    x_mix  = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (y2 - y1) * (x2 - x1) / (H * W)
    return x_mix, y, y[perm], lam_actual


def smooth_ce(out, ya, yb=None, lam=1.0, smoothing=0.1, nc=10):
    log_p = F.log_softmax(out, dim=1)
    def nll(t):
        with torch.no_grad():
            s = torch.full_like(log_p, smoothing / (nc - 1))
            s.scatter_(1, t.unsqueeze(1), 1.0 - smoothing)
        return -(s * log_p).sum(1).mean()
    return lam * nll(ya) + (1.0 - lam) * nll(yb) if yb is not None else nll(ya)


def make_scheduler(optimizer, warmup, total):
    def lr_fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        t = (ep - warmup) / (total - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * t))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


# ════════════════════════════════════════════════════════════════════════════
# 10. TRAIN / EVAL LOOPS
# ════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, epoch, warmup, cutmix_p, log_grads=False):
    model.train()
    tot_loss = correct = total = 0

    for bidx, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)

        if epoch > warmup and np.random.random() < cutmix_p:
            imgs, la, lb, lam = cutmix(imgs, labels)
            out  = model(imgs)
            loss = smooth_ce(out, la, lb, lam)
        else:
            out  = model(imgs)
            loss = smooth_ce(out, labels)

        optimizer.zero_grad()
        loss.backward()

        if bidx == 0 and log_grads:
            print("\n  [GRADIENT CHECK — Epoch 1, Batch 0]")
            dead = True
            for name, p in model.named_parameters():
                if p.grad is not None and p.grad.abs().mean() > 1e-9 and 'weight' in name:
                    print(f"  ✓  {name:50s}  grad={p.grad.abs().mean().item():.2e}")
                    dead = False
            print("  ✗  ALL GRADIENTS ZERO" if dead else "  ✓  Gradients flowing.")
            print()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        tot_loss += loss.item()
        _, pred = out.max(1)
        correct += pred.eq(labels).sum().item()
        total   += labels.size(0)

    return tot_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        _, pred = model(imgs).max(1)
        correct += pred.eq(labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total


# ════════════════════════════════════════════════════════════════════════════
# 11. MAIN
# ════════════════════════════════════════════════════════════════════════════
def main():
    EPOCHS      = 200
    BATCH_SIZE  = 128
    LR          = 3e-3
    WD          = 1e-4
    WARMUP      = 15
    CUTMIX_PROB = 0.5
    BASE_CH     = 64
    N_BLOCKS    = 4
    DIAG_EVERY  = 25
    SAVE_NAME   = "hybrid_qam16_qpsk_resnet_best.pth"

    print("=" * 72)
    print("  HYBRID QUANTIZED ResNet — CIFAR-10")
    print(f"  Input        : 16-QAM  levels={[f'{v:.4f}' for v in _QAM16_LEVELS.tolist()]}")
    print(f"  Weights/Acts : QPSK    levels=[{-QPSK_LEVEL:.4f}, {+QPSK_LEVEL:.4f}]")
    print(f"  Architecture : base_ch={BASE_CH}  blocks/stage={N_BLOCKS}")
    print(f"  Float parts  : first conv + skip projections + classifier head (~1%)")
    print(f"  Epochs       : {EPOCHS}  |  LR: {LR}  |  Warmup: {WARMUP}")
    print(f"  Target       : 82%+")
    print("=" * 72)

    train_loader, test_loader = get_dataloaders(BATCH_SIZE)

    model = HybridResNet(base_ch=BASE_CH, n_blocks=N_BLOCKS).to(device)
    total, fp, qp = model.param_summary()
    print(f"\nParameters : {total:,} total")
    print(f"  Float     : {fp:,}  ({100*fp/total:.1f}%)")
    print(f"  Quantized : {qp:,}  ({100*qp/total:.1f}%)")

    # Sanity check input quantizer
    sample_imgs, _ = next(iter(test_loader))
    print("\n[Input Quantizer Sanity Check — must see exactly 4 unique levels]")
    model.input_q.verify(sample_imgs[:64].to(device))

    # Separate param groups: no weight decay on BN / bias / scale params
    decay, nodecay = [], []
    for name, p in model.named_parameters():
        if 'bn' in name or name.endswith('.bias') or 'alpha' in name or 'beta' in name:
            nodecay.append(p)
        else:
            decay.append(p)

    optimizer = torch.optim.AdamW(
        [{'params': decay,   'weight_decay': WD},
         {'params': nodecay, 'weight_decay': 0.0}],
        lr=LR, betas=(0.9, 0.999)
    )
    scheduler = make_scheduler(optimizer, WARMUP, EPOCHS)

    best_acc = 0.0
    print(f"\nTraining...\n")

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer,
            epoch, WARMUP, CUTMIX_PROB,
            log_grads=(epoch == 1)
        )
        te_acc = evaluate(model, test_loader)
        scheduler.step()

        is_best = te_acc > best_acc
        if is_best:
            best_acc = te_acc
            torch.save(model.state_dict(), SAVE_NAME)

        lr_now = optimizer.param_groups[0]['lr']
        star   = "  ★ BEST" if is_best else ""
        print(f"Ep {epoch:3d}/{EPOCHS} | "
              f"Loss {tr_loss:.4f} | "
              f"Train {tr_acc:.2f}% | "
              f"Test {te_acc:.2f}% | "
              f"Best {best_acc:.2f}% | "
              f"LR {lr_now:.2e}"
              f"{star}")

        if epoch % DIAG_EVERY == 0:
            print(f"\n  [DIAGNOSTICS — Epoch {epoch}]")
            print(f"  {'Layer':40s}  unique_wq  snap_err   alpha_mean")
            for name, m in model.named_modules():
                if isinstance(m, QPSKConv2d):
                    wt = torch.tanh(m.weight) * QPSK_MAX
                    wq = qpsk_snap(wt)
                    with torch.no_grad():
                        n_u = wq.unique().numel()
                        err = (wt - wq).abs().mean().item()
                        alp = m.alpha.abs().mean().item()
                    print(f"  {name:40s}  {n_u}/2       {err:.4f}     {alp:.3f}")
            print()

    print(f"\n{'='*72}")
    print(f"  Training complete!")
    print(f"  Best test accuracy : {best_acc:.2f}%")
    print(f"  Target (82%+)      : {'✓ MET' if best_acc >= 82.0 else '✗ below target'}")
    print(f"  Saved to           : {SAVE_NAME}")
    print(f"{'='*72}")


if __name__ == "__main__":
    main()